In [ ]:
!python -m pip install -U python-woc pandas matplotlib
# please clear the output of this cell in your notebook before checking it in

In [ ]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd

# Creates the WoC client
woc = WocMapsRemote(base_url="https://worldofcode.org/api/")

# Your 10 assigned GitHub projects
projects = [
    'aspuru-guzik-group/tequila',
    'inasafe/inasafe',
    'Parallel-NetCDF/PnetCDF',
    'tbereau/auto_martini',
    'AMReX-Codes/IAMR',
    'cmusatyalab/openface',
    'Jstacs/Jstacs',
    'csgillespie/efficientR',
    'datmo/datmo',
    'voxelmorph/voxelmorph'
]

list_df_commits = []

for project in projects:
    # Convert GitHub repository name to WoC project ID
    project_wocid = project.lower().replace('/', '_', 2)

    # Get all commit SHA1s for this project
    commits = woc.get_values('p2c', project_wocid)

    df_project = pd.DataFrame(commits, columns=["sha1"])

    # Keep both the original project name and WoC project ID
    df_project['project'] = project
    df_project['project_wocid'] = project_wocid

    list_df_commits.append(df_project)

# Combine all 10 projects
df = pd.concat(list_df_commits, ignore_index=True)

# Save so we don't have to retrieve the project commit lists again
df.to_csv('df_commits.csv', index=False)

df.head(10)

In [ ]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    project_info = df[df['sha1'] == commit_sha].iloc[0]
    # flatten commit objects
    commit_data.append({
          'project': project_info['project'],
          'project_wocid': project_info['project_wocid'],
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

In [ ]:
import time

# Resume from the batch where the previous run stopped
start_batch = 3710

for i in tqdm(range(start_batch, len(chunks))):

    chunk = chunks[i]

    try:
        res, err = woc.get_values_many(
            'commit.tch',
            chunk.to_list()
        )

        res = {k: v[0] for k, v in res.items()}

        if err:
            print(f"Got Errors in batch {i}:", err)

        for commit_sha, commit in res.items():

            project_info = df[df['sha1'] == commit_sha].iloc[0]

            commit_data.append({
                'project': project_info['project'],
                'project_wocid': project_info['project_wocid'],
                'commit': commit_sha,
                'tree': commit[0],
                'parent': list(commit[1]),
                'author': commit[2][0],
                'author_time': int(commit[2][1]),
                'author_tz': commit[2][2],
                'committer': commit[3][0],
                'committer_time': int(commit[3][1]),
                'committer_tz': commit[3][2],
                'message': commit[4],
            })

    except Exception as e:
        print(f"ERROR in batch {i}: {type(e).__name__}: {e}")
        print("Continuing to the next batch...")

    # Respect WoC rate limit
    time.sleep(1)

print("Finished resume.")
print("Total commit records retrieved:", len(commit_data))

In [ ]:
# Create the dataframe required for the assignment

dfinf = pd.DataFrame({
    'project_wocid': [k['project_wocid'] for k in commit_data],
    'commit_sha1': [k['commit'] for k in commit_data],
    'author': [k['author'] for k in commit_data],
    'time': [k['author_time'] for k in commit_data],
    'commit message': [k['message'] for k in commit_data]
})


In [ ]:
# check if it has the right content
dfinf.head(1)

In [ ]:
dfinf.to_csv('hawad_project_summary.csv', index=False, sep=';')
print("Saved. Rows:", len(dfinf))

# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github

## Project statistics (WoC + GitHub)

In [11]:
import pandas as pd
import requests

projects = [
    'aspuru-guzik-group/tequila', 'inasafe/inasafe', 'Parallel-NetCDF/PnetCDF',
    'tbereau/auto_martini', 'AMReX-Codes/IAMR', 'cmusatyalab/openface',
    'Jstacs/Jstacs', 'csgillespie/efficientR', 'datmo/datmo', 'voxelmorph/voxelmorph'
]

dfinf = pd.read_csv('hawad_project_summary.csv', sep=';')
print(len(dfinf), dfinf.columns.tolist())

48397 ['project_wocid', 'commit_sha1', 'author', 'time', 'commit message']


In [12]:
woc_summary = (
    dfinf.groupby('project_wocid')
    .agg(commits=('commit_sha1', 'nunique'),
         authors=('author', 'nunique'),
         max_time=('time', 'max'),
         min_time=('time', 'min'))
    .reset_index()
)
woc_summary['max_time'] = pd.to_datetime(woc_summary['max_time'], unit='s')
woc_summary['min_time'] = pd.to_datetime(woc_summary['min_time'], unit='s')
woc_summary

,project_wocid,commits,authors,max_time,min_time
0,amrex-codes_iamr,5194,89,2025-04-29 01:43:10,1997-07-08 23:08:02
1,aspuru-guzik-group_tequila,2660,127,2025-11-04 10:06:19,2019-05-08 19:01:04
2,cmusatyalab_openface,1239,79,2024-10-04 12:59:06,2015-09-18 20:57:25
3,csgillespie_efficientr,1927,71,2025-03-10 17:00:53,2016-01-11 17:17:17
4,datmo_datmo,1093,8,2025-06-23 06:29:41,2018-04-01 01:02:04
5,inasafe_inasafe,20930,108,2025-06-24 10:23:36,2012-01-09 10:38:49
6,jstacs_jstacs,4450,15,2025-10-24 13:22:50,2008-08-27 14:26:43
7,parallel-netcdf_pnetcdf,9339,30,2025-10-27 06:09:45,2003-02-05 16:46:07
8,tbereau_auto_martini,409,14,2025-05-28 12:02:51,2014-07-30 13:42:31
9,voxelmorph_voxelmorph,1156,76,2025-10-10 00:15:00,2018-05-14 16:29:28


In [13]:
rows = []
for p in projects:
    repo = requests.get(f"https://api.github.com/repos/{p}").json()
    commit = requests.get(f"https://api.github.com/repos/{p}/commits?per_page=1").json()
    rows.append({
        'project_wocid': p.lower().replace('/', '_', 2),
        'stars': repo['stargazers_count'],
        'forks': repo['forks_count'],
        'last_commit': commit[0]['commit']['committer']['date'],
    })

github_info = pd.DataFrame(rows)
github_info

,project_wocid,stars,forks,last_commit
0,aspuru-guzik-group_tequila,442,120,2026-05-28T15:35:34Z
1,inasafe_inasafe,283,131,2025-05-21T12:09:51Z
2,parallel-netcdf_pnetcdf,109,36,2026-09-27T21:10:57Z
3,tbereau_auto_martini,76,23,2025-05-28T12:02:51Z
4,amrex-codes_iamr,105,99,2026-09-28T00:13:10Z
5,cmusatyalab_openface,15444,3559,2026-07-18T15:27:00Z
6,jstacs_jstacs,23,5,2026-03-12T09:01:26Z
7,csgillespie_efficientr,740,381,2025-02-27T11:20:29Z
8,datmo_datmo,342,29,2025-06-23T06:29:41Z
9,voxelmorph_voxelmorph,2754,643,2026-09-11T20:00:03Z


In [14]:
final_report = woc_summary.merge(github_info, on='project_wocid', how='left')
final_report

,project_wocid,commits,authors,max_time,min_time,stars,forks,last_commit
0,amrex-codes_iamr,5194,89,2025-04-29 01:43:10,1997-07-08 23:08:02,105,99,2026-09-28T00:13:10Z
1,aspuru-guzik-group_tequila,2660,127,2025-11-04 10:06:19,2019-05-08 19:01:04,442,120,2026-05-28T15:35:34Z
2,cmusatyalab_openface,1239,79,2024-10-04 12:59:06,2015-09-18 20:57:25,15444,3559,2026-07-18T15:27:00Z
3,csgillespie_efficientr,1927,71,2025-03-10 17:00:53,2016-01-11 17:17:17,740,381,2025-02-27T11:20:29Z
4,datmo_datmo,1093,8,2025-06-23 06:29:41,2018-04-01 01:02:04,342,29,2025-06-23T06:29:41Z
5,inasafe_inasafe,20930,108,2025-06-24 10:23:36,2012-01-09 10:38:49,283,131,2025-05-21T12:09:51Z
6,jstacs_jstacs,4450,15,2025-10-24 13:22:50,2008-08-27 14:26:43,23,5,2026-03-12T09:01:26Z
7,parallel-netcdf_pnetcdf,9339,30,2025-10-27 06:09:45,2003-02-05 16:46:07,109,36,2026-09-27T21:10:57Z
8,tbereau_auto_martini,409,14,2025-05-28 12:02:51,2014-07-30 13:42:31,76,23,2025-05-28T12:02:51Z
9,voxelmorph_voxelmorph,1156,76,2025-10-10 00:15:00,2018-05-14 16:29:28,2754,643,2026-09-11T20:00:03Z


## Results

| Project | Stars | Forks | Last commit (GitHub) | WoC commits | WoC authors | WoC max time | WoC min time |
|---|---|---|---|---|---|---|---|
| aspuru-guzik-group/tequila | 442 | 120 | 2026-05-28 | 2660 | 127 | 2025-11-04 10:06:19 | 2019-05-08 19:01:04 |
| inasafe/inasafe | 283 | 131 | 2025-05-21 | 20930 | 108 | 2025-06-24 10:23:36 | 2012-01-09 10:38:49 |
| Parallel-NetCDF/PnetCDF | 109 | 36 | 2026-09-27 | 9339 | 30 | 2025-10-27 06:09:45 | 2003-02-05 16:46:07 |
| tbereau/auto_martini | 76 | 23 | 2025-05-28 | 409 | 14 | 2025-05-28 12:02:51 | 2014-07-30 13:42:31 |
| AMReX-Codes/IAMR | 105 | 99 | 2026-09-28 | 5194 | 89 | 2025-04-29 01:43:10 | 1997-07-08 23:08:02 |
| cmusatyalab/openface | 15444 | 3559 | 2026-07-18 | 1239 | 79 | 2024-10-04 12:59:06 | 2015-09-18 20:57:25 |
| Jstacs/Jstacs | 23 | 5 | 2026-03-12 | 4450 | 15 | 2025-10-24 13:22:50 | 2008-08-27 14:26:43 |
| csgillespie/efficientR | 740 | 381 | 2025-02-27 | 1927 | 71 | 2025-03-10 17:00:53 | 2016-01-11 17:17:17 |
| datmo/datmo | 342 | 29 | 2025-06-23 | 1093 | 8 | 2025-06-23 06:29:41 | 2018-04-01 01:02:04 |
| voxelmorph/voxelmorph | 2754 | 643 | 2026-09-11 | 1156 | 76 | 2025-10-10 00:15:00 | 2018-05-14 16:29:28 |